# 04 - Scheduling: deterministic baseline vs QpiAI Opt

**Purpose:** Solve the wave schedule exactly with the deterministic baseline and check the same MILP with QpiAI Opt's offline tools.

**Inputs:** Phase 2 problem; qpiai-opt (offline classify / inspect / feasibility only)

**Outputs:** schedule, constraint checks, Opt model checks

**Status labels:** Baseline VERIFIED. QpiAI Opt offline checks VERIFIED. Authenticated `solve(mode='local')` BLOCKED (not authorized / no key). No platform, submit() or QPU.

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
from cryptoshift.planning.planner import build_problem, schedule, to_tasks
p = build_problem(); s = schedule(p)
print('feasible', s['feasible'], '| objective', s['objective'], '| waves', s['waves'])
print('dependency violations:', s['dependency_violations'], '| constraint violations:', s['constraint_violations'])

In [ ]:
from cryptoshift.optimization.member3_qpiai import opt_wave_model_checks
tasks, _ = to_tasks(p)
o = opt_wave_model_checks(tasks, p['config']['constraints'], s['task_to_wave'])
{k: o[k] for k in ('sdk_versions','problem_class','binary_variables','constraints','baseline_feasible_in_opt_model','baseline_objective_recomputed','authenticated_local_solve')}

**Plain language.** Each service gets exactly one wave; a provider must move no later than its consumers; each wave holds at most 2 services and 85 minutes (assumptions). The baseline tries every assignment and keeps the cheapest valid one, so its answer is optimal for this small problem. QpiAI Opt accepted the same model as a MILP and confirmed the baseline answer is feasible in it. Solving it with QpiAI needs an API key and owner authorization, which this run did not have, so there is no QpiAI-vs-baseline objective comparison yet.